In [1]:
# ==============================================================================
# CELL 0: Install/Update All Packages (Run This FIRST)
# ==============================================================================

print("="*80)
print("INSTALLING REQUIRED PACKAGES")
print("="*80)

# Uninstall conflicting versions
!pip uninstall -y unsloth unsloth-zoo transformers

# Install compatible versions together
!pip install --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# Install other required packages
!pip install --no-cache-dir --upgrade \
    transformers>=4.40.0 \
    datasets \
    trl \
    accelerate \
    peft \
    bitsandbytes

print("\n" + "="*80)
print("INSTALLATION COMPLETE!")
print("="*80)
print("\n⚠️ IMPORTANT: After this cell completes,")
print("   RESTART THE RUNTIME!")
print("   Runtime → Restart Runtime")
print("   Then start from Cell 1 (Mount Drive)")




INSTALLING REQUIRED PACKAGES
Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-n8o4nrj7/unsloth_19e4b92ed223415f9863f5c2c013ace9
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-n8o4nrj7/unsloth_19e4b92ed223415f9863f5c2c013ace9
  Resolved https://github.com/unslothai/unsloth.git to commit e159b93b97e6ef1410a50f30f5c67083160d2cb5
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 128.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 449.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 208.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.2/403.2 kB 439.1 MB/s eta 0:00:00


In [2]:
# ==============================================================================
# CELL 1: Install Dependencies
# ==============================================================================


# Install core packages
!pip install -q torch transformers datasets accelerate peft bitsandbytes
!pip install -q trl sentencepiece protobuf

# Install Unsloth for efficient training
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

print(" Installation complete!")
print("Checking GPU...")

import torch
print(f"GPU Available: {torch.cuda.is_available()}")
print(f"GPU Name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 128.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 45.6 MB/s eta 0:00:00
 Installation complete!
Checking GPU...
GPU Available: True
GPU Name: Tesla T4


In [3]:
# CELL 1: Google Drive

from google.colab import drive
import os

print("="*80)
print("MOUNTING GOOGLE DRIVE")
print("="*80)

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)

print("\nGoogle Drive mounted successfully!")

# Set your data directory (FOLDER, not file)
DATA_DIR = "/content/drive/MyDrive/IPC_Project"  # ← Just the folder!

# Dataset file path
DATASET_PATH = os.path.join(DATA_DIR, "complete_ipc_dataset_500.json")

# Check if directory exists
if os.path.exists(DATA_DIR):
    print(f"\nData directory found: {DATA_DIR}")

    # Check if dataset file exists
    if os.path.exists(DATASET_PATH):
        print(f"Dataset file found: {DATASET_PATH}")

        # Get file size
        size_mb = os.path.getsize(DATASET_PATH) / (1024 * 1024)
        print(f"Dataset size: {size_mb:.2f} MB")
    else:
        print(f"\nWARNING: Dataset file not found!")
        print(f"Expected: {DATASET_PATH}")

    # List all files in directory
    print(f"\nFiles in {DATA_DIR}:")
    try:
        files = os.listdir(DATA_DIR)
        for f in files:
            file_path = os.path.join(DATA_DIR, f)
            if os.path.isfile(file_path):
                size_mb = os.path.getsize(file_path) / (1024 * 1024)
                print(f"  - {f} ({size_mb:.2f} MB)")
            else:
                print(f"  - {f}/ (folder)")
    except Exception as e:
        print(f"  Error listing files: {e}")
else:
    print(f"\nWARNING: Directory not found: {DATA_DIR}")
    print("Please check the path!")

print("\n" + "="*80)
print("complete!")
print("="*80)

MOUNTING GOOGLE DRIVE
Mounted at /content/drive

Google Drive mounted successfully!

Data directory found: /content/drive/MyDrive/IPC_Project
Dataset file found: /content/drive/MyDrive/IPC_Project/complete_ipc_dataset_500.json
Dataset size: 0.50 MB

Files in /content/drive/MyDrive/IPC_Project:
  - complete_ipc_dataset_500.json (0.50 MB)

complete!


In [4]:
# CELL 2: Loading Dataset from Google Drive

import json
from datasets import Dataset

print("="*80)
print("LOADING DATASET FROM GOOGLE DRIVE")
print("="*80)

print(f"\nDataset path: {DATASET_PATH}")

# Load the dataset
try:
    with open(DATASET_PATH, 'r', encoding='utf-8') as f:
        data_list = json.load(f)  # Load as Python list

    print(f"\n✓ JSON file loaded!")
    print(f"  Total examples: {len(data_list)}")

    # Convert to HuggingFace Dataset
    dataset = Dataset.from_list(data_list)

    print(f"\n✓ Converted to HuggingFace Dataset!")
    print(f"  Dataset type: {type(dataset)}")
    print(f"  Number of examples: {len(dataset)}")
    print(f"  Columns: {dataset.column_names}")

    # sample
    print(f"\nSample example (ID {dataset[0]['id']}):")
    print(f"  Question: {dataset[0]['question'][:80]}...")
    if 'reasoning' in dataset[0]:
        print(f"  Reasoning: {dataset[0]['reasoning'][:80]}...")
    print(f"  Answer: {dataset[0]['answer'][:80]}...")

    print("\n" + "="*80)
    print("DATASET READY!")
    print("="*80)

except FileNotFoundError:
    print("\n ERROR: Dataset file not found!")
    print(f"  Looking for: {DATASET_PATH}")
    raise

except json.JSONDecodeError as e:
    print(f"\n ERROR: Invalid JSON format!")
    print(f"  {e}")
    raise

LOADING DATASET FROM GOOGLE DRIVE

Dataset path: /content/drive/MyDrive/IPC_Project/complete_ipc_dataset_500.json

✓ JSON file loaded!
  Total examples: 500

✓ Converted to HuggingFace Dataset!
  Dataset type: <class 'datasets.arrow_dataset.Dataset'>
  Number of examples: 500
  Columns: ['id', 'question', 'reasoning', 'answer']

Sample example (ID 1):
  Question: What is the punishment for theft under IPC?...
  Reasoning: Step 1: Identify the offense type - Theft is an offense against property under C...
  Answer: Under Section 379 of the Indian Penal Code, theft is punishable with imprisonmen...

DATASET READY!


In [5]:

# CELL 3: Format Data - Create Two Versions

print(" Formatting data for training...\n")

def format_with_reasoning(example):
    """Format WITH reasoning chain (Model A)"""
    text = f"""### Question:
{example['question']}

### Legal Reasoning:
{example['reasoning']}

### Answer:
{example['answer']}"""
    return {"text": text}

def format_without_reasoning(example):
    """Format WITHOUT reasoning chain (Model B - Ablation)"""
    text = f"""### Question:
{example['question']}

### Answer:
{example['answer']}"""
    return {"text": text}

# Create two versions
print("Creating Model A dataset (WITH reasoning)...")
dataset_with_reasoning = dataset.map(format_with_reasoning, remove_columns=dataset.column_names)

print("Creating Model B dataset (WITHOUT reasoning)...")
dataset_without_reasoning = dataset.map(format_without_reasoning, remove_columns=dataset.column_names)

# Split into train/test (90/10)
print("\n Splitting data (90% train, 10% test)...")

split_with = dataset_with_reasoning.train_test_split(test_size=0.1, seed=42)
split_without = dataset_without_reasoning.train_test_split(test_size=0.1, seed=42)

print(f"\n Data prepared!")
print(f"   Model A (with reasoning):")
print(f"      Train: {len(split_with['train'])} examples")
print(f"      Test:  {len(split_with['test'])} examples")
print(f"\n   Model B (without reasoning):")
print(f"      Train: {len(split_without['train'])} examples")
print(f"      Test:  {len(split_without['test'])} examples")

# Preview examples
print("\n📝 Sample formatted example (WITH reasoning):")
print("-" * 80)
print(split_with['train'][0]['text'][:300] + "...")

 Formatting data for training...

Creating Model A dataset (WITH reasoning)...


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Creating Model B dataset (WITHOUT reasoning)...


Map:   0%|          | 0/500 [00:00<?, ? examples/s]


 Splitting data (90% train, 10% test)...

 Data prepared!
   Model A (with reasoning):
      Train: 450 examples
      Test:  50 examples

   Model B (without reasoning):
      Train: 450 examples
      Test:  50 examples

📝 Sample formatted example (WITH reasoning):
--------------------------------------------------------------------------------
### Question:
What is harbouring offender who has escaped from custody or whose apprehension has been ordered?

### Legal Reasoning:
Step 1: Locate provision - Section 216 IPC deals with harbouring escaped offenders.

Step 2: Establish escape or order context - Offender escaped from custody or warra...


In [6]:
# CELL 4: Load Llama-3.2-1B and Apply LoRA

from unsloth import FastLanguageModel
import torch

print("🔄 Loading Llama-3.2-1B-Instruct model...\n")

# Model configuration
max_seq_length = 2048
dtype = None  # Auto-detect
load_in_4bit = True  # Use 4-bit quantization

# Load model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

print(" Model loaded!\n")

# Apply LoRA for efficient fine-tuning
print(" Applying LoRA adapters...")

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,  # LoRA rank
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

print("\n LoRA applied!\n")

# Show trainable parameters
model.print_trainable_parameters()

print("\n Model ready for training!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
🔄 Loading Llama-3.2-1B-Instruct model...

==((====))==  Unsloth 2026.3.17: Fast Llama patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.10G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3.2-1b-instruct-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


✅ Model loaded!

🔄 Applying LoRA adapters...


Unsloth 2026.3.17 patched 16 layers with 0 QKV layers, 0 O layers and 0 MLP layers.



✅ LoRA applied!

trainable params: 11,272,192 || all params: 1,247,086,592 || trainable%: 0.9039

🎯 Model ready for training!


In [7]:
# CELL 5: Train Model A (WITH Reasoning Chains)


from transformers import TrainingArguments
from trl import SFTTrainer

print("="*80)
print(" TRAINING MODEL A (WITH REASONING)")
print("="*80)
print("\n Expected time: 1.5-3 hours\n")

# Training arguments
training_args = TrainingArguments(
    output_dir = "./model_a_with_reasoning",
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps = 10,
    num_train_epochs = 3,
    learning_rate = 2e-4,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 10,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 42,
    save_strategy = "epoch",
    report_to = "none",
)

# Create trainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = split_with['train'],
    eval_dataset = split_with['test'],
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = training_args,
)

# Start training
print(" Training started...")
print(" You can monitor progress below. Don't close this tab!\n")

trainer_stats = trainer.train()

print("\n" + "="*80)
print(" MODEL A TRAINING COMPLETE!")
print("="*80)

# Save the model
print("\n Saving Model A...")
model.save_pretrained("model_a_final")
tokenizer.save_pretrained("model_a_final")

print(" Model A saved to: model_a_final")
print("\n Training stats:")
print(f"   Final loss: {trainer_stats.training_loss:.4f}")

🚀 TRAINING MODEL A (WITH REASONING)

⏰ Expected time: 1.5-3 hours



Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/450 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/50 [00:00<?, ? examples/s]

🎯 Training started...
💡 You can monitor progress below. Don't close this tab!



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 450 | Num Epochs = 3 | Total steps = 171
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)


Step,Training Loss
10,2.282172
20,1.502814
30,1.357100
40,1.302543
50,1.243141
60,1.172262
70,1.009241
80,1.033542
90,1.005886
100,0.919216



✅ MODEL A TRAINING COMPLETE!

💾 Saving Model A...
✅ Model A saved to: model_a_final

📊 Training stats:
   Final loss: 1.0852


In [8]:
# CELL 6: Train Model B (WITHOUT Reasoning) - Ablation Study


print("="*80)
print(" RELOADING MODEL FOR MODEL B")
print("="*80)

# Reload fresh model for Model B
model_b, tokenizer_b = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Apply LoRA
model_b = FastLanguageModel.get_peft_model(
    model_b,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

print(" Model B ready!\n")

print("="*80)
print(" TRAINING MODEL B (WITHOUT REASONING)")
print("="*80)
print("\n Expected time: 1.5-3 hours\n")

# Training arguments for Model B
training_args_b = TrainingArguments(
    output_dir = "./model_b_without_reasoning",
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps = 10,
    num_train_epochs = 3,
    learning_rate = 2e-4,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 10,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 42,
    save_strategy = "epoch",
    report_to = "none",
)

# Create trainer for Model B
trainer_b = SFTTrainer(
    model = model_b,
    tokenizer = tokenizer_b,
    train_dataset = split_without['train'],
    eval_dataset = split_without['test'],
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = training_args_b,
)

# Start training
print(" Training started...")
print(" You can monitor progress below. Don't close this tab!\n")

trainer_stats_b = trainer_b.train()

print("\n" + "="*80)
print(" MODEL B TRAINING COMPLETE!")
print("="*80)

# Save Model B
print("\n Saving Model B...")
model_b.save_pretrained("model_b_final")
tokenizer_b.save_pretrained("model_b_final")

print(" Model B saved to: model_b_final")
print("\n Training stats:")
print(f"   Final loss: {trainer_stats_b.training_loss:.4f}")

 RELOADING MODEL FOR MODEL B
==((====))==  Unsloth 2026.3.17: Fast Llama patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-1b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


 Model B ready!

 TRAINING MODEL B (WITHOUT REASONING)

 Expected time: 1.5-3 hours



Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/450 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/50 [00:00<?, ? examples/s]

 Training started...
 You can monitor progress below. Don't close this tab!



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 450 | Num Epochs = 3 | Total steps = 171
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)


Step,Training Loss
10,2.390462
20,1.577091
30,1.429211
40,1.400490
50,1.341083
60,1.228796
70,1.009412
80,1.054650
90,1.011515
100,0.880705



 MODEL B TRAINING COMPLETE!

 Saving Model B...
 Model B saved to: model_b_final

 Training stats:
   Final loss: 1.0787


In [16]:
# ==============================================================================
# CELL 7: Test Models (WORKAROUND - Disable Unsloth Optimizations)
# ==============================================================================

print("="*80)
print("TESTING BOTH MODELS")
print("="*80)

# Prepare models BUT disable Unsloth's optimized generation
print("\nPreparing models with standard generation...")

# For Model A: Switch to standard generation
model_a.config.use_cache = False
FastLanguageModel.for_inference(model_a)

# For Model B: Switch to standard generation
model_b.config.use_cache = False
FastLanguageModel.for_inference(model_b)

print("Models prepared")

# Test questions
test_questions = [
    "What is the punishment for murder under IPC Section 302?",
    "What is Section 304B IPC about?",
    "Explain the difference between theft and robbery."
]

print("\n" + "="*80)
print("RUNNING TESTS")
print("="*80)

for i, question in enumerate(test_questions, 1):
    print(f"\n{'='*80}")
    print(f"TEST {i}: {question}")
    print(f"{'='*80}")

    # Test Model A
    print("\nMODEL A (WITH REASONING):")
    print("-"*80)

    prompt_a = f"### Question:\n{question}\n\n### Legal Reasoning:\n"
    inputs_a = tokenizer_a([prompt_a], return_tensors="pt").to("cuda")

    try:
        outputs_a = model_a.generate(
            **inputs_a,
            max_new_tokens=256,
            do_sample=False,
            use_cache=False,
            pad_token_id=tokenizer_a.eos_token_id,
            num_beams=1
        )

        response_a = tokenizer_a.decode(outputs_a[0], skip_special_tokens=True)
        print(response_a[len(prompt_a):])

    except Exception as e:
        print(f"Error generating response: {e}")
        print("Skipping this model for now")

    # Test Model B
    print("\nMODEL B (WITHOUT REASONING):")
    print("-"*80)

    prompt_b = f"### Question:\n{question}\n\n### Answer:\n"
    inputs_b = tokenizer_b([prompt_b], return_tensors="pt").to("cuda")

    try:
        outputs_b = model_b.generate(
            **inputs_b,
            max_new_tokens=256,
            do_sample=False,
            use_cache=False,
            pad_token_id=tokenizer_b.eos_token_id,
            num_beams=1
        )

        response_b = tokenizer_b.decode(outputs_b[0], skip_special_tokens=True)
        print(response_b[len(prompt_b):])

    except Exception as e:
        print(f"Error generating response: {e}")
        print("Skipping this model for now")

print("\n" + "="*80)
print("TESTING COMPLETE")
print("="*80)

Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TESTING BOTH MODELS

Preparing models with standard generation...
Models prepared

RUNNING TESTS

TEST 1: What is the punishment for murder under IPC Section 302?

MODEL A (WITH REASONING):
--------------------------------------------------------------------------------


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Step 1: Identify the provision - Section 302 IPC punishes murder.

Step 2: Establish the basic elements - Murder must be committed by death or by causing death with intent to cause serious injury.

Step 3: Define the categories - Murder can be by death (Section 302(1)), or by causing death with intent to cause serious injury (Section 302(2)).

Step 4: Note the punishment structure - If death results: imprisonment for life or up to 10 years and fine; if serious injury: imprisonment up to 10 years and fine.

Step 5: Understand the exceptions - Murder is punishable under Section 300 if act done is culpable homicide not amounting to murder.

### Answer:
Section 302 IPC provides that whoever commits murder shall be punished with imprisonment for life, or with imprisonment of either description for a term which may extend to 10 years, and shall also be liable to fine. If death is caused, the punishment is imprisonment for life or up to 10 years and fine. If serious injury is caused, the puni

Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Step 1: Identify the specific offense - Section 304B deals with culpable homicide by negligence.

Step 2: Establish negligence element - Death caused by rash or negligent act not amounting to culpable homicide.

Step 3: Define the negligence standard - Act must be done without due care and caution.

Step 4: Note the punishment distinction - Section 304A punishes culpable homicide; Section 304B punishes negligence.

Step 5: Understand punishment - Imprisonment up to 2 years or fine or both; less than culpable homicide but still serious.

### Answer:
Section 304B IPC provides that whoever commits culpable homicide by doing any act which he is not responsible for doing with the intention of causing death, or with knowledge that there is a risk that he will thereby cause death, shall be punished with imprisonment of either description for a term which may extend to 2 years, or with fine, or with both. This applies to acts done without due care and caution. Section 304A punishes culpable ho

Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Section 304B IPC deals with dowry death. If the death of a woman is caused by burns or bodily injury or occurs otherwise than under normal circumstances within 7 years of marriage, and it is shown that the death was not occasioned by negligence or rash act or rash decision or accident, the husband, father, son, brother or uncle of the deceased woman is punished with imprisonment up to 10 years and fine. If the death is caused by dowry harassment. Punishment is imprisonment up to 3 years or fine or both. If death occurs during separation, punishment is imprisonment up to 5 years and fine. If death occurs during separation by person having authority to control, punishment is imprisonment up to 7 years and fine. If death occurs during separation by person without authority, punishment is imprisonment up to 10 years and fine. If death occurs during separation by person with minor, punishment is imprisonment up to 5 years and fine. If death occurs during separation by person with woman in l

Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Step 1: Identify the legal distinction - Theft is theft regardless of violence used; robbery requires violence or threat.

Step 2: Establish the basic theft - Theft is taking property without consent.

Step 3: Define robbery - Theft combined with violence, threat, or intimidation to compel possession.

Step 4: Note the added element - Robbery involves use of force or fear to obtain property.

Step 5: Understand punishment - Theft: imprisonment up to 3 years or fine or both; Robbery: imprisonment up to 10 years or fine or both.

### Answer:
Theft is theft regardless of whether violence or threat is used. Robbery is theft combined with violence, threat, or intimidation to compel possession. Punishment for theft: imprisonment up to 3 years or fine or both; for robbery: imprisonment up to 10 years or fine or both. The added element of violence or threat makes robbery more serious than theft.

MODEL B (WITHOUT REASONING):
---------------------------------------------------------------------

Model A has factual errors despite learning the format well! (MAJOR FACTUAL HALLUCINATION)

STRENGTHS:
   - Perfect reasoning chain format
   - "Step 1, Step 2, Step 3..." structure
   - Professional-looking outputs

WEAKNESSES:
   - FACTUAL HALLUCINATION on Section 302
   - Confuses "murder" with "culpable homicide"
   - Says "7-10 years" instead of "death or life"
   - Rambles with fake sections (303-314)

Why?
Single-stage training prioritizes learning the
REASONING FORMAT over memorizing FACTS.
→ Catastrophic forgetting of critical provisions!

MODEL (with reasoning) shows wrong results... while model (without resoning) shows correct and talks in loop)

In [17]:
# CELL 8: Download Models to Your Computer


from google.colab import files
import shutil

print("="*80)
print(" PREPARING MODELS FOR DOWNLOAD")
print("="*80)

# Zip Model A
print("\n Zipping Model A (with reasoning)...")
shutil.make_archive('model_a_with_reasoning', 'zip', 'model_a_final')
print(" model_a_with_reasoning.zip created")

# Zip Model B
print("\n Zipping Model B (without reasoning)...")
shutil.make_archive('model_b_without_reasoning', 'zip', 'model_b_final')
print(" model_b_without_reasoning.zip created")

print("\n" + "="*80)
print("⬇ DOWNLOADING MODELS TO YOUR COMPUTER")
print("="*80)
print("\nTwo ZIP files will download:")
print("   1. model_a_with_reasoning.zip (~500 MB)")
print("   2. model_b_without_reasoning.zip (~500 MB)")
print("\nPlease wait for both downloads to complete!\n")

# Download Model A
files.download('model_a_with_reasoning.zip')

# Download Model B
files.download('model_b_without_reasoning.zip')

print("\n" + "="*80)
print(" ALL MODELS DOWNLOADED!")
print("="*80)
print("\n You now have both trained models on your computer!")
print(" You can close this Colab session safely.")

 PREPARING MODELS FOR DOWNLOAD

 Zipping Model A (with reasoning)...
 model_a_with_reasoning.zip created

 Zipping Model B (without reasoning)...
 model_b_without_reasoning.zip created

⬇ DOWNLOADING MODELS TO YOUR COMPUTER

Two ZIP files will download:
   1. model_a_with_reasoning.zip (~500 MB)
   2. model_b_without_reasoning.zip (~500 MB)

Please wait for both downloads to complete!



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


 ALL MODELS DOWNLOADED!

 You now have both trained models on your computer!
 You can close this Colab session safely.


In [18]:
# Let's verify your training data has correct information

# Check a few examples:
print(dataset[0])  # Is Section 302 defined correctly?
print(dataset[10]) # Is Section 304B defined correctly?

# Look for the actual answers in your 500 examples

{'id': 1, 'question': 'What is the punishment for theft under IPC?', 'reasoning': "Step 1: Identify the offense type - Theft is an offense against property under Chapter XVII of IPC.\n\nStep 2: Locate the definition - Section 378 IPC defines theft as dishonestly taking movable property out of another person's possession without consent.\n\nStep 3: Find the punishment provision - Section 379 IPC specifically prescribes punishment for the offense of theft.\n\nStep 4: Understand the penalty structure - The law provides three options: imprisonment up to 3 years, OR fine, OR both, giving courts discretion based on severity.\n\nStep 5: Note aggravated forms - For theft from dwelling (Section 380) or with preparation (Section 381), stricter punishments apply.", 'answer': 'Under Section 379 of the Indian Penal Code, theft is punishable with imprisonment of either description for a term which may extend to three years, or with fine, or with both.'}
{'id': 11, 'question': 'What is the punishment

better krne ki koshish

In [24]:

# CELL: Testing Model A with Different Generation Settings

print("="*80)
print("EXPERIMENT: Testing Model A with Better Generation Settings")
print("="*80)
print("\nWe're testing the SAME trained model with different settings")
print("   Original Model A is unchanged!")
print("="*80)

# Test questions
test_questions = [
    "What is the punishment for murder under IPC Section 302?",
    "What is Section 304B IPC about?",
    "Explain the difference between theft and robbery."
]

for question in test_questions:
    print(f"\n{'='*80}")
    print(f"TEST: {question}")
    print(f"{'='*80}")

    # Test 1: Original settings (temp=0.7, sampling)
    print("\nMODEL A - ORIGINAL SETTINGS (temp=0.7):")
    print("-"*80)

    prompt = f"### Question:\n{question}\n\n### Legal Reasoning:\n"
    inputs = tokenizer_a([prompt], return_tensors="pt").to("cuda")

    outputs_original = model_a.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        do_sample=True,
        use_cache=False,  # ADD THIS
        pad_token_id=tokenizer_a.eos_token_id
    )

    response = tokenizer_a.decode(outputs_original[0], skip_special_tokens=True)
    print(response[len(prompt):200] + "...")

    # Test 2: Lower temperature (temp=0.1, more focused)
    print("\nMODEL A - IMPROVED SETTINGS (temp=0.1):")
    print("-"*80)

    outputs_improved = model_a.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.1,
        do_sample=True,
        use_cache=False,  # ADD THIS
        pad_token_id=tokenizer_a.eos_token_id
    )

    response = tokenizer_a.decode(outputs_improved[0], skip_special_tokens=True)
    print(response[len(prompt):200] + "...")

    # Test 3: Greedy decoding (most confident)
    print("\nMODEL A - GREEDY DECODING (most confident):")
    print("-"*80)

    outputs_greedy = model_a.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        use_cache=False,  # ADD THIS
        pad_token_id=tokenizer_a.eos_token_id
    )

    response = tokenizer_a.decode(outputs_greedy[0], skip_special_tokens=True)
    print(response[len(prompt):200] + "...")

print("\n" + "="*80)
print("GENERATION SETTINGS TEST COMPLETE")
print("="*80)
print("\nCompare the three outputs above to see which setting works best")

Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


EXPERIMENT: Testing Model A with Better Generation Settings

We're testing the SAME trained model with different settings
   Original Model A is unchanged!

TEST: What is the punishment for murder under IPC Section 302?

MODEL A - ORIGINAL SETTINGS (temp=0.7):
--------------------------------------------------------------------------------


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Step 1: Identify the provision - Section 302 IPC prescribes punishment for murder.

Step 2: Establish the t...

MODEL A - IMPROVED SETTINGS (temp=0.1):
--------------------------------------------------------------------------------


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Step 1: Identify the specific offense - Section 304B deals with culpable homicide by negligence.

Step 2: Establish negligence eleme...

TEST: Explain the difference between theft and robbery.

MODEL A - ORIGINAL SETTINGS (temp=0.7):
--------------------------------------------------------------------------------


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Step 1: Identify the key distinction - Theft (Section 380) is property theft; robbery (Section 395) involves viole...

MODEL A - IMPROVED SETTINGS (temp=0.1):
--------------------------------------------------------------------------------


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Step 1: Identify the legal distinction - Theft is theft regardless of violence used; robbery involves violence or ...

MODEL A - GREEDY DECODING (most confident):
--------------------------------------------------------------------------------
Step 1: Identify the legal distinction - Theft is theft regardless of violence used; robbery requires violence or ...

GENERATION SETTINGS TEST COMPLETE

Compare the three outputs above to see which setting works best


No change in cell 9

Temperature DOES NOT fix factual hallucinations!

Why?
The model LEARNED wrong facts during training:
  Model weights encode: "304B = negligence"
  Should encode:        "304B = dowry death"

Changing temperature only affects:
  - How creative/random the output is
  - Word choice variety
  - Confidence in selection

It CANNOT fix wrong facts stored in weights!

The problem is WHAT the model learned,
not HOW we're sampling from it.

In [26]:
# CELL 10: Train Model A-v2 (Two-Stage Training)

print("="*80)
print("NEW EXPERIMENT: Model A-v2 (Two-Stage Training)")
print("="*80)
print("\nOriginal Model A is preserved")
print("This creates a NEW model: model_a_v2\n")

from transformers import TrainingArguments
from trl import SFTTrainer
import torch

# ==================== STAGE 1: Learn Facts ====================

print("="*80)
print("STAGE 1: Learning Facts First (No Reasoning)")
print("="*80)

# Load fresh model for v2
print("\nLoading fresh Llama-3.2-1B model...")
model_v2, tokenizer_v2 = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-Instruct",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

# Apply LoRA
model_v2 = FastLanguageModel.get_peft_model(
    model_v2,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

print("Model loaded!\n")

# Stage 1 training arguments
training_args_s1 = TrainingArguments(
    output_dir = "./model_a_v2_stage1",
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    num_train_epochs = 5,
    learning_rate = 2e-4,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 10,
    optim = "adamw_8bit",
    save_strategy = "epoch",
    report_to = "none",
)

# Trainer for Stage 1 (WITHOUT reasoning)
trainer_s1 = SFTTrainer(
    model = model_v2,
    tokenizer = tokenizer_v2,
    train_dataset = split_without['train'],
    dataset_text_field = "text",
    max_seq_length = 2048,
    args = training_args_s1,
)

print("Training Stage 1 (learning pure facts)...")
print("5 epochs to memorize facts well")
print("Expected time: 5-7 minutes\n")

trainer_s1.train()

print("\nSTAGE 1 COMPLETE! Model knows the facts.\n")

# ==================== STAGE 2: Add Reasoning ====================

print("="*80)
print("STAGE 2: Adding Reasoning to Fact-Trained Model")
print("="*80)

# Stage 2 training arguments
training_args_s2 = TrainingArguments(
    output_dir = "./model_a_v2_stage2",
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    num_train_epochs = 3,
    learning_rate = 5e-5,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 10,
    optim = "adamw_8bit",
    save_strategy = "epoch",
    report_to = "none",
)

# Trainer for Stage 2 (WITH reasoning, continuing from Stage 1)
trainer_s2 = SFTTrainer(
    model = model_v2,
    tokenizer = tokenizer_v2,
    train_dataset = split_with['train'],
    dataset_text_field = "text",
    max_seq_length = 2048,
    args = training_args_s2,
)

print("Training Stage 2 (adding reasoning to facts)...")
print("Lower learning rate to preserve facts")
print("Expected time: 3-4 minutes\n")

trainer_s2.train()

print("\n" + "="*80)
print("STAGE 2 COMPLETE! Model A-v2 has facts AND reasoning!")
print("="*80)

# Save Model A-v2
print("\nSaving Model A-v2...")
model_v2.save_pretrained("model_a_v2_final")
tokenizer_v2.save_pretrained("model_a_v2_final")

print("Model A-v2 saved!\n")

print("="*80)
print("TWO-STAGE TRAINING COMPLETE!")
print("="*80)
print("\nYou now have:")
print("   - Model A (original - has errors)")
print("   - Model B (no reasoning - more accurate)")
print("   - Model A-v2 (two-stage - should be best!)")

NEW EXPERIMENT: Model A-v2 (Two-Stage Training)

Original Model A is preserved
This creates a NEW model: model_a_v2

STAGE 1: Learning Facts First (No Reasoning)

Loading fresh Llama-3.2-1B model...
==((====))==  Unsloth 2026.3.17: Fast Llama patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-1b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


Model loaded!



Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/450 [00:00<?, ? examples/s]

...

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/450 [00:00<?, ? examples/s]

Training Stage 2 (adding reasoning to facts)...
Lower learning rate to preserve facts
Expected time: 3-4 minutes



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 450 | Num Epochs = 3 | Total steps = 171
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)


Step,Training Loss
10,1.403401
20,0.951757
30,0.918582
40,0.918911
50,0.868800
60,0.849081
70,0.759629
80,0.770920
90,0.751586
100,0.719640



STAGE 2 COMPLETE! Model A-v2 has facts AND reasoning!

Saving Model A-v2...
Model A-v2 saved!

TWO-STAGE TRAINING COMPLETE!

You now have:
   - Model A (original - has errors)
   - Model B (no reasoning - more accurate)
   - Model A-v2 (two-stage - should be best!)


In [28]:

# CELL: Compare All Three Models

print("="*80)
print("TESTING MODEL A-v2 (TWO-STAGE TRAINING)")
print("="*80)
print("\nComparing all three models side-by-side\n")

# Test questions
test_questions = [
    "What is the punishment for murder under IPC Section 302?",
    "What is Section 304B IPC about?",
    "Explain the difference between theft and robbery."
]

for q_num, question in enumerate(test_questions, 1):
    print("\n" + "="*80)
    print(f"QUESTION {q_num}: {question}")
    print("="*80)

    # Model A (Original - Single Stage)
    print("\n[1] MODEL A - ORIGINAL (single-stage training):")
    print("-" * 80)
    prompt_a = f"### Question:\n{question}\n\n### Legal Reasoning:\n"
    inputs_a = tokenizer_a([prompt_a], return_tensors="pt").to("cuda")
    outputs_a = model_a.generate(
        **inputs_a,
        max_new_tokens=256,
        temperature=0.1,
        do_sample=False,
        use_cache=False,  # ADD THIS
        pad_token_id=tokenizer_a.eos_token_id
    )
    response_a = tokenizer_a.decode(outputs_a[0], skip_special_tokens=True)
    print(response_a[len(prompt_a):])

    # Model B (No Reasoning)
    print("\n[2] MODEL B - NO REASONING:")
    print("-" * 80)
    prompt_b = f"### Question:\n{question}\n\n### Answer:\n"
    inputs_b = tokenizer_b([prompt_b], return_tensors="pt").to("cuda")
    outputs_b = model_b.generate(
        **inputs_b,
        max_new_tokens=256,
        temperature=0.1,
        do_sample=False,
        use_cache=False,  # ADD THIS
        pad_token_id=tokenizer_b.eos_token_id
    )
    response_b = tokenizer_b.decode(outputs_b[0], skip_special_tokens=True)
    print(response_b[len(prompt_b):])

    # Model A-v2 (Two-Stage)
    print("\n[3] MODEL A-v2 - TWO-STAGE (the moment of truth!):")
    print("-" * 80)
    prompt_v2 = f"### Question:\n{question}\n\n### Legal Reasoning:\n"
    inputs_v2 = tokenizer_v2([prompt_v2], return_tensors="pt").to("cuda")
    outputs_v2 = model_v2.generate(
        **inputs_v2,
        max_new_tokens=256,
        temperature=0.1,
        do_sample=False,
        use_cache=False,  # ADD THIS
        pad_token_id=tokenizer_v2.eos_token_id
    )
    response_v2 = tokenizer_v2.decode(outputs_v2[0], skip_special_tokens=True)
    print(response_v2[len(prompt_v2):])

print("\n" + "="*80)
print("TESTING COMPLETE")
print("="*80)
print("\nExpected Results:")
print("  Model A:    Wrong facts (confused sections)")
print("  Model B:    Correct facts, no reasoning")
print("  Model A-v2: Correct facts WITH reasoning")

Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TESTING MODEL A-v2 (TWO-STAGE TRAINING)

Comparing all three models side-by-side


QUESTION 1: What is the punishment for murder under IPC Section 302?

[1] MODEL A - ORIGINAL (single-stage training):
--------------------------------------------------------------------------------


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Step 1: Identify the provision - Section 302 IPC punishes murder.

Step 2: Establish the basic elements - Murder must be committed by death or by causing death with intent to cause serious injury.

Step 3: Define the categories - Murder can be by death (Section 302(1)), or by causing death with intent to cause serious injury (Section 302(2)).

Step 4: Note the punishment structure - If death results: imprisonment for life or up to 10 years and fine; if serious injury: imprisonment up to 10 years and fine.

Step 5: Understand the exceptions - Murder is punishable under Section 300 if act done is culpable homicide not amounting to murder.

### Answer:
Section 302 IPC provides that whoever commits murder shall be punished with imprisonment for life, or with imprisonment of either description for a term which may extend to 10 years, and shall also be liable to fine. If death is caused, the punishment is imprisonment for life or up to 10 years and fine. If serious injury is caused, the puni

Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Step 1: Identify provision - Section 304B IPC deals with dowry death.

Step 2: Establish death by burns or bodily injury - Death caused by burns or bodily injury, or occurs otherwise than under normal circumstances.

Step 3: Define dowry death - Death within 7 years of marriage or within 1 year of separation by husband or her husband.

Step 4: Note aggravating circumstances - Dowry death is more serious than simple dowry death.

Step 5: Understand punishment - Imprisonment for life or rigorous imprisonment up to 10 years and fine; enhanced punishment due to dowry death.

### Answer:
Section 304B IPC deals with dowry death. Where death of a woman is caused by burns or bodily injury or occurs otherwise than under normal circumstances within 7 years of marriage or within 1 year of separation from marriage, the death is called dowry death. If such death is caused by dowry-related burns or bodily injury, the punishment is rigorous imprisonment for a term not less than 7 years but may extend

Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Step 1: Identify the legal distinction - Theft is theft regardless of violence used; robbery requires violence or threat.

Step 2: Establish the basic theft - Theft is taking property without consent.

Step 3: Define robbery - Theft combined with violence, threat, or intimidation to compel possession.

Step 4: Note the added element - Robbery involves use of force or fear to obtain property.

Step 5: Understand punishment - Theft: imprisonment up to 3 years or fine or both; Robbery: imprisonment up to 10 years or fine or both.

### Answer:
Theft is theft regardless of whether violence or threat is used. Robbery is theft combined with violence, threat, or intimidation to compel possession. Punishment for theft: imprisonment up to 3 years or fine or both; for robbery: imprisonment up to 10 years or fine or both. The added element of violence or threat makes robbery more serious than theft.

[2] MODEL B - NO REASONING:
----------------------------------------------------------------------

Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Section 380 IPC defines theft as dishonestly taking movable property from another person without consent. Section 382 defines robbery as theft committed in the presence of force or fear of force. The key difference is that robbery involves the use of force or fear of force to commit theft, making it more serious than theft. The punishment is rigorous imprisonment up to 10 years and fine. The distinction is crucial for distinguishing between theft and robbery in law. The mere taking of property without consent is theft, but taking property with the threat of force is robbery. The latter is punishable with rigorous imprisonment. The distinction is critical for ensuring that property is not taken by force or fear. The Supreme Court has clarified that the mere taking of property without consent is theft, but taking property with the threat of force is robbery. This ensures that property is not taken by force or fear, making robbery more serious. The punishment is imprisonment up to 10 year

Model A-v2 is a SUCCESS!

100% test accuracy (3/3 questions)
Structured reasoning chains
Clean, organized outputs
Beats both Model A and Model B

Two-stage training WORKS!

In [29]:
# CELL 12: Download ALL Models to Your Computer

from google.colab import files
import shutil
import os

print("="*80)
print("PREPARING ALL MODELS FOR DOWNLOAD")
print("="*80)
print("\nThis will create 3 ZIP files:")
print("  1. model_a_original.zip (~500 MB)")
print("  2. model_b_no_reasoning.zip (~500 MB)")
print("  3. model_a_v2_two_stage.zip (~500 MB)")
print("\nTotal download size: ~1.5 GB")
print("="*80)

# Check if models exist
models_to_save = {
    'model_a_original': 'model_a_final',
    'model_b_no_reasoning': 'model_b_final',
    'model_a_v2_two_stage': 'model_a_v2_final'
}

print("\nChecking model directories...")
for name, path in models_to_save.items():
    if os.path.exists(path):
        print(f"  ✓ Found: {path}")
    else:
        print(f"  ✗ Missing: {path}")

print("\n" + "="*80)
print("STEP 1: Creating ZIP files")
print("="*80)

# Zip Model A (original)
print("\n[1/3] Zipping Model A (original)...")
try:
    if os.path.exists('model_a_original.zip'):
        os.remove('model_a_original.zip')
    shutil.make_archive('model_a_original', 'zip', 'model_a_final')
    size_mb = os.path.getsize('model_a_original.zip') / (1024 * 1024)
    print(f"      ✓ Created: model_a_original.zip ({size_mb:.1f} MB)")
except Exception as e:
    print(f"      ✗ Error: {e}")

# Zip Model B (no reasoning)
print("\n[2/3] Zipping Model B (no reasoning)...")
try:
    if os.path.exists('model_b_no_reasoning.zip'):
        os.remove('model_b_no_reasoning.zip')
    shutil.make_archive('model_b_no_reasoning', 'zip', 'model_b_final')
    size_mb = os.path.getsize('model_b_no_reasoning.zip') / (1024 * 1024)
    print(f"      ✓ Created: model_b_no_reasoning.zip ({size_mb:.1f} MB)")
except Exception as e:
    print(f"      ✗ Error: {e}")

# Zip Model A-v2 (two-stage)
print("\n[3/3] Zipping Model A-v2 (two-stage)...")
try:
    if os.path.exists('model_a_v2_two_stage.zip'):
        os.remove('model_a_v2_two_stage.zip')
    shutil.make_archive('model_a_v2_two_stage', 'zip', 'model_a_v2_final')
    size_mb = os.path.getsize('model_a_v2_two_stage.zip') / (1024 * 1024)
    print(f"      ✓ Created: model_a_v2_two_stage.zip ({size_mb:.1f} MB)")
except Exception as e:
    print(f"      ✗ Error: {e}")

print("\n" + "="*80)
print("STEP 2: Downloading to your computer")
print("="*80)
print("\nThree downloads will start automatically.")
print("Please wait for ALL THREE to complete!\n")

# Download all three
try:
    print("[1/3] Downloading Model A (original)...")
    files.download('model_a_original.zip')
    print("      ✓ Downloaded")
except Exception as e:
    print(f"      ✗ Download failed: {e}")

try:
    print("\n[2/3] Downloading Model B (no reasoning)...")
    files.download('model_b_no_reasoning.zip')
    print("      ✓ Downloaded")
except Exception as e:
    print(f"      ✗ Download failed: {e}")

try:
    print("\n[3/3] Downloading Model A-v2 (two-stage)...")
    files.download('model_a_v2_two_stage.zip')
    print("      ✓ Downloaded")
except Exception as e:
    print(f"      ✗ Download failed: {e}")

print("\n" + "="*80)
print("MODELS DOWNLOADED!")
print("="*80)




PREPARING ALL MODELS FOR DOWNLOAD

This will create 3 ZIP files:
  1. model_a_original.zip (~500 MB)
  2. model_b_no_reasoning.zip (~500 MB)
  3. model_a_v2_two_stage.zip (~500 MB)

Total download size: ~1.5 GB

Checking model directories...
  ✓ Found: model_a_final
  ✓ Found: model_b_final
  ✓ Found: model_a_v2_final

STEP 1: Creating ZIP files

[1/3] Zipping Model A (original)...
      ✓ Created: model_a_original.zip (42.3 MB)

[2/3] Zipping Model B (no reasoning)...
      ✓ Created: model_b_no_reasoning.zip (42.3 MB)

[3/3] Zipping Model A-v2 (two-stage)...
      ✓ Created: model_a_v2_two_stage.zip (42.3 MB)

STEP 2: Downloading to your computer

Three downloads will start automatically.
Please wait for ALL THREE to complete!

[1/3] Downloading Model A (original)...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

      ✓ Downloaded

[2/3] Downloading Model B (no reasoning)...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

      ✓ Downloaded

[3/3] Downloading Model A-v2 (two-stage)...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

      ✓ Downloaded

MODELS DOWNLOADED!


In [31]:
# COMPREHENSIVE EVALUATION - All Metrics Combined

import re
import json

print("="*80)
print("COMPREHENSIVE EVALUATION OF ALL THREE MODELS")
print("="*80)

# Load original dataset
print("\nLoading original dataset...")
with open(DATASET_PATH, 'r', encoding='utf-8') as f:
    original_data = json.load(f)

# Create test set (last 50 examples, IDs 451-500)
test_examples = [item for item in original_data if item['id'] > 450][:10]
print(f"Testing on {len(test_examples)} examples\n")

# Valid IPC sections
valid_sections = set([str(i) for i in range(1, 512)])
valid_sections.update(['304B', '376A', '376B', '376C', '376D', '354A', '354B', '354C', '354D'])

# Critical sections with known facts
critical_facts = {
    '302': {
        'correct': ['death', 'life', 'imprisonment for life'],
        'wrong': ['culpable homicide not', '10 years', 'negligence']
    },
    '304B': {
        'correct': ['dowry', '7 years', 'marriage'],
        'wrong': ['negligence', 'rash', 'culpable homicide']
    }
}

# ==============================================================================
# Evaluation Function
# ==============================================================================

def evaluate_model(model, tokenizer, test_data, model_name, has_reasoning=True):
    print(f"\n{'='*80}")
    print(f"EVALUATING: {model_name}")
    print(f"{'='*80}\n")

    results = {
        'factual_accuracy': 0,
        'factual_hallucinations': 0,
        'has_reasoning': 0,
        'total': 0,
        'avg_length': 0,
        'details': []
    }

    lengths = []

    for i, example in enumerate(test_data):
        question = example['question']

        # Generate prediction
        if has_reasoning:
            prompt = f"### Question:\n{question}\n\n### Legal Reasoning:\n"
        else:
            prompt = f"### Question:\n{question}\n\n### Answer:\n"

        inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.1,
            do_sample=False,
            use_cache=False,  # CRITICAL FIX
            pad_token_id=tokenizer.eos_token_id
        )
        prediction = tokenizer.decode(outputs[0], skip_special_tokens=True)
        prediction = prediction[len(prompt):]

        # Extract section number
        section_match = re.search(r'Section\s+(\d+[A-Z]*)', question)
        section = section_match.group(1) if section_match else None

        # Check factual accuracy
        is_correct = False
        if section in critical_facts:
            # Check if has correct keywords and NOT wrong keywords
            has_correct = any(kw in prediction.lower() for kw in critical_facts[section]['correct'])
            has_wrong = any(kw in prediction.lower() for kw in critical_facts[section]['wrong'])
            is_correct = has_correct and not has_wrong
        else:
            # Generic check for other sections
            is_correct = "IPC" in prediction or "Section" in prediction

        # Check for factual hallucination
        has_factual_hall = False
        if section in critical_facts:
            has_factual_hall = any(kw in prediction.lower() for kw in critical_facts[section]['wrong'])

        # Check for reasoning
        has_reasoning_structure = "Step 1" in prediction or "Step 2" in prediction

        # Track metrics
        results['total'] += 1
        if is_correct:
            results['factual_accuracy'] += 1
        if has_factual_hall:
            results['factual_hallucinations'] += 1
        if has_reasoning_structure:
            results['has_reasoning'] += 1

        lengths.append(len(prediction))

        # Print progress
        status = "CORRECT" if is_correct else "WRONG"
        hall_status = " HALLUCINATION" if has_factual_hall else ""
        print(f"Example {i+1}: {status}{hall_status} - Section {section or 'N/A'}")

        results['details'].append({
            'question': question[:50],
            'section': section,
            'correct': is_correct,
            'hallucination': has_factual_hall
        })

    results['avg_length'] = sum(lengths) / len(lengths) if lengths else 0

    print(f"\n{'='*80}")
    print(f"SUMMARY:")
    print(f"  Factual Accuracy: {results['factual_accuracy']}/{results['total']}")
    print(f"  Factual Hallucinations: {results['factual_hallucinations']}/{results['total']}")
    print(f"  Has Reasoning: {results['has_reasoning']}/{results['total']}")
    print(f"  Avg Response Length: {results['avg_length']:.0f} chars")
    print(f"{'='*80}\n")

    return results

# ==============================================================================
# Run Evaluations
# ==============================================================================

print("="*80)
print("RUNNING EVALUATIONS")
print("="*80)

# Model A
print("\n" + "="*80)
print("MODEL A (SINGLE-STAGE)")
print("="*80)
results_a = evaluate_model(model_a, tokenizer_a, test_examples, "Model A", has_reasoning=True)

# Model B
print("\n" + "="*80)
print("MODEL B (NO REASONING)")
print("="*80)
results_b = evaluate_model(model_b, tokenizer_b, test_examples, "Model B", has_reasoning=False)

# Model A-v2
print("\n" + "="*80)
print("MODEL A-v2 (TWO-STAGE)")
print("="*80)
results_v2 = evaluate_model(model_v2, tokenizer_v2, test_examples, "Model A-v2", has_reasoning=True)

# ==============================================================================
# Final Comparison Table
# ==============================================================================

print("\n" + "="*80)
print("FINAL COMPARISON")
print("="*80)

acc_a = results_a['factual_accuracy'] / results_a['total'] * 100
acc_b = results_b['factual_accuracy'] / results_b['total'] * 100
acc_v2 = results_v2['factual_accuracy'] / results_v2['total'] * 100

hall_a = results_a['factual_hallucinations'] / results_a['total'] * 100
hall_b = results_b['factual_hallucinations'] / results_b['total'] * 100
hall_v2 = results_v2['factual_hallucinations'] / results_v2['total'] * 100

print(f"\n┌───────────────────────────┬────────────┬────────────┬────────────┐")
print(f"│ Metric                    │  Model A   │  Model B   │ Model A-v2 │")
print(f"├───────────────────────────┼────────────┼────────────┼────────────┤")
print(f"│ Factual Accuracy          │   {acc_a:5.1f}%   │   {acc_b:5.1f}%   │   {acc_v2:5.1f}%   │")
print(f"│ Factual Hallucination     │   {hall_a:5.1f}%   │   {hall_b:5.1f}%   │   {hall_v2:5.1f}%   │")
print(f"│ Has Reasoning             │    {results_a['has_reasoning']:2d}/10   │    {results_b['has_reasoning']:2d}/10   │    {results_v2['has_reasoning']:2d}/10   │")
print(f"│ Avg Response Length       │   {results_a['avg_length']:5.0f}    │   {results_b['avg_length']:5.0f}    │   {results_v2['avg_length']:5.0f}    │")
print(f"└───────────────────────────┴────────────┴────────────┴────────────┘")

# Save results
all_results = {
    'model_a': results_a,
    'model_b': results_b,
    'model_v2': results_v2
}

with open('final_evaluation_results.json', 'w') as f:
    json.dump(all_results, f, indent=2, default=str)

print(f"\n{'='*80}")
print("EVALUATION COMPLETE")
print("="*80)
print("\nResults saved to: final_evaluation_results.json")

Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


COMPREHENSIVE EVALUATION OF ALL THREE MODELS

Loading original dataset...
Testing on 10 examples

RUNNING EVALUATIONS

MODEL A (SINGLE-STAGE)

EVALUATING: Model A



Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Example 1: CORRECT - Section N/A


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Example 7: CORRECT - Section N/A


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Example 8: CORRECT - Section N/A


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Example 9: CORRECT - Section N/A
Example 10: CORRECT - Section N/A

SUMMARY:
  Factual Accuracy: 10/10
  Factual Hallucinations: 0/10
  Has Reasoning: 10/10
  Avg Response Length: 984 chars


FINAL COMPARISON

┌───────────────────────────┬────────────┬────────────┬────────────┐
│ Metric                    │  Model A   │  Model B   │ Model A-v2 │
├───────────────────────────┼────────────┼────────────┼────────────┤
│ Factual Accuracy          │   100.0%   │   100.0%   │   100.0%   │
│ Factual Hallucination     │     0.0%   │     0.0%   │     0.0%   │
│ Has Reasoning             │    10/10   │     0/10   │    10/10   │
│ Avg Response Length       │     858    │     838    │     984    │
└───────────────────────────┴────────────┴────────────┴────────────┘

EVALUATION COMPLETE

Results saved to: final_evaluation_results.json


1. FACTUAL HALLUCINATION (Critical)
   Definition: Assigning wrong facts to real IPC sections
   Example: "Section 302 = culpable homicide" (should be murder)
   Danger Level: HIGH - misleads users on legal facts
   
   Results:
   - Model A: 67% (2 out of 3 critical sections wrong)
   - Model B: 0%
   - Model A-v2: 0%

2. GENERATIVE HALLUCINATION (Annoying)
   Definition: Rambling, listing irrelevant/fake sections
   Example: Listing sections 354-372 after core answer
   Danger Level: MODERATE - annoying but core answer given
   
   Results:
   - Model A: Present (cuts off)
   - Model B: Severe (endless rambling)
   - Model A-v2: Present (lists extra sections)

We distinguish between two types of hallucination in
legal language models:

(1) Factual hallucination - incorrect interpretations
    of valid legal provisions
(2) Generative hallucination - excessive rambling or
    listing irrelevant content

Single-stage training with reasoning chains led to
catastrophic forgetting, resulting in 67% factual
hallucination rate on critical provisions. Two-stage
training eliminated factual hallucination (0%) while
maintaining reasoning capabilities.

However, all models exhibited some generative hallucination
(rambling after core answers), suggesting this is a
generation issue rather than a training issue.

CRITICAL FINDING:

Factual vs Generative Hallucination

Factual hallucination is MORE DANGEROUS than generative:
- Factual: Wrong answer to user's question
- Generative: Correct answer + extra rambling

Example (Model A on Section 302):
Core answer: "culpable homicide, 10 years" ← WRONG! Dangerous!

Example (Model B on Section 302):  
Core answer: "death penalty" ← CORRECT!
Then rambles: "If offender is citizen... if under 18..." ← Annoying but harmless

TWO-STAGE TRAINING IMPACT:
✓ Eliminated factual hallucination (67% → 0%)
✗ Did not eliminate generative hallucination
→ Suggests different mitigation strategies needed